# Step 7b · Backcasting 2026 → 2050: cut car CO$_2$ by 10 % at least cost (Grenoble)

**Backcasting question:** what is the cheapest yearly investment plan in cycling infrastructure (2027–2050) such that the **cumulative car CO$_2$ emissions 2027–2050 are at least 10 % below** the reference trajectory without new investment?

This notebook follows the optimal control problem of `book.tex` (chapter *Backcasting*) and the car-side equations of the *Cycling circulation, simplified model* (`book.tex`, `bicycle_model.tex`):

$$
\min_{u(t)} \; J=\sum_{t=2027}^{T} C^B(t),\qquad C^B(t)=u(t)\,(1+\tau)^{-(t-2026)}
\quad\text{s.t.}\quad \mathcal{E}(T)\le(1-0.10)\,\bar{\mathcal{E}}(T),\quad 0\le u(t)\le u^{\max}
$$

| Block | Equation | Source |
|---|---|---|
| Bike share | $P_B(t)=f\big(\sum_{s\le t}u(s)\big)$, $f$ = **MLCC lookup** (1,000 states, logistic model of N and weighted LCC) | replaces the first-order $k\,u_B(P_B^\infty-P_B)$ law by the network-based lookup |
| Car share | $P_C(t)=\bar P_C(t)-\xi\,(P_B(t)-P_B(0))$ | book, eq. after (pb) |
| Car distance | $d_C(t)=\bar d_C\,\dfrac{\bar P_C(t)-r_{BC}\,P_C^B(t)}{P_C(t)}$, $r_{BC}=d_B/\bar d_C$ | book |
| Car traffic | $G_C(t)=\bar G_C(t)\,\dfrac{d_C(t)}{\bar d_C}\dfrac{P_C(t)}{\bar P_C(t)}$, $\bar G_C=\text{trips}\cdot\bar P_C\,\bar d_C/\omega$ | book |
| Emissions | $E(t)=\epsilon(t)\,G_C(t)$, $\mathcal{E}(t)=\mathcal{E}(t-1)+E(t)$ | book, CO$_2$ emissions |
| Cost | $C^B(t)=u(t)(1+\tau)^{-t}$ (€ allocated, discounted) | book, cost of the cycling policy |

**Differences with notebook `07a_backcasting_2050_target_35pct.ipynb`:**
* $\xi$ and $r_{BC}$ come from the scenario table of `bicycle_model.tex` (the original assumed $\xi=1$ and that every removed car trip had the *average car* length, i.e. $r_{BC}=1$; that case is kept as the scenario *"Original notebook"*).
* Spending is **discounted** ($\tau$), so the optimizer trades "spend later (cheaper)" against "reduce emissions earlier".
* The optimization is solved **exactly on the 1,000 MLCC states** (dynamic programming + Lagrange multiplier on the CO$_2$ budget), so every plan is a network actually produced by MLCC; no smooth surrogate is needed. Infeasible targets are reported with the maximum reduction that is achievable.
* The 35 % bike share in 2050 is an **optional** extra constraint (`TERMINAL_SHARE_TARGET`).

| | |
|---|---|
| **Input** | `results/mlcc_lookup/grenoble_lookup_infrastructure_only.csv` (step 5) |
| **Output** | `results/backcasting/co2_target/` |
| **Run time** | ~1 min |

> `book.tex` and `bicycle_model.tex` are the colleague's model documents (first-order bike-share model, backcasting chapter). They are cited for the equations but are **not included** in this public repository.

## 1. Settings

In [1]:
# Setup: find the repository root and make the `cyclelts` package importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyclelts").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from cyclelts import config

# ---- Lookup produced by notebooks/05_mlcc_lookup_table.ipynb ----
LOOKUP_DIR = config.LOOKUP_DIR
LOOKUP_CSV = LOOKUP_DIR / "grenoble_lookup_infrastructure_only.csv"
OUTPUT_DIR = config.BACKCASTING_DIR / "co2_target"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Backcasting problem -----------------------------------------------------------------------
START_YEAR, END_YEAR = 2026, 2050        # 2026 = observed state; decisions u(t) for 2027..END_YEAR
CO2_REDUCTION_TARGET = 0.10              # cumulative 2027..END_YEAR emissions <= (1-target) * reference
TERMINAL_SHARE_TARGET = None             # e.g. 35.0 to also require P_B(END_YEAR) >= 35 %
ANNUAL_ALLOCATION_CAP_EUR = 2_000_000.0  # u_max (illustrative)
DISCOUNT_RATE = 0.032                    # tau, public discount rate (illustrative; 0 = no discounting)

# ---- Mode-share model (same calibration as the lookup) ----------------------------------------
W_N, W_C = 0.40, 0.60
ALPHA, BETA, P_MAX = -3.40675, 5.01498, 0.50

# ---- Car / emission model (same values as the original notebook) ------------------------------
TRIPS_PER_YEAR = 20_000_000              # annual person-trips of the modelled population
CAR_SHARE_REF = 0.25                     # \bar P_C(t): reference car share (constant)
CAR_DISTANCE_KM = 8.0                    # \bar d_C
CAR_OCCUPANCY = 1.3                      # omega
EMISSION_FACTOR_2026 = 150.0             # epsilon (gCO2 / veh-km) in 2026
EMISSION_FACTOR_END = 150.0              # epsilon in END_YEAR (linear in between); lower = fleet electrification

# ---- Bicycle -> car parameters: scenario table of bicycle_model.tex ---------------------------
# xi   : car share lost per point of bike share gained
# r_BC : distance of a bike trip / average car distance
PARAMETER_SETS = {
    "Conservative":      dict(xi=0.30, r_BC=0.27),
    "Intermediate":      dict(xi=0.50, r_BC=0.35),
    "Optimistic":        dict(xi=0.65, r_BC=0.40),
    "Original notebook": dict(xi=1.00, r_BC=1.00),
}
SCENARIO = "Intermediate"                # main scenario for the detailed plan and plots

## 2. MLCC lookup: budget → network metrics → bike share

In [2]:
raw = pd.read_csv(LOOKUP_CSV).sort_values("budget_allocated_eur").reset_index(drop=True)
b = raw["budget_allocated_eur"].to_numpy(float)
N = raw["normalized_network_length"].to_numpy(float)
C = raw["weighted_lcc_normalized"].to_numpy(float)
P = 100 * P_MAX / (1 + np.exp(-(ALPHA + BETA * (W_N * N + W_C * C))))   # % bike (two-wheel) share

step = np.diff(b)
assert np.isclose(b[0], 0) and np.allclose(step, step[0]), "Expected a uniform budget grid starting at 0"
assert np.all(np.diff(P) >= -1e-9), "Share must not decrease with budget"
STEP_EUR = float(step[0])
K_MAX = int(np.floor(ANNUAL_ALLOCATION_CAP_EUR / STEP_EUR + 1e-9))   # lookup steps allowed per year
P0 = float(P[0])

years = np.arange(START_YEAR + 1, END_YEAR + 1)
T = len(years)
discount = (1 + DISCOUNT_RATE) ** -(years - START_YEAR).astype(float)
emission_factor = np.interp(years, [START_YEAR, END_YEAR], [EMISSION_FACTOR_2026, EMISSION_FACTOR_END])

print(f"{len(b)} MLCC states, €0 – €{b[-1]:,.0f} (step €{STEP_EUR:,.0f}); "
      f"annual cap = {K_MAX} steps = €{K_MAX*STEP_EUR:,.0f}")
print(f"Bike share: {P0:.3f} % (2026) → {P[-1]:.3f} % at the end of the lookup")
print(f"Decision years: {years[0]}–{years[-1]} ({T} years); discount rate {DISCOUNT_RATE:.1%}")

1000 MLCC states, €0 – €38,595,734 (step €38,634); annual cap = 51 steps = €1,970,353
Bike share: 22.820 % (2026) → 39.791 % at the end of the lookup
Decision years: 2027–2050 (24 years); discount rate 3.2%


## 3. Car model and emissions (book equations)

In [3]:
def car_model(bike_share_percent, xi, r_BC):
    # P_C, d_C, G_C (veh-km/year) for a bike share, using the book's equations.
    dPB = (np.asarray(bike_share_percent, float) - P0) / 100.0
    PcB = xi * dPB                                     # car share lost to the bike
    Pc = CAR_SHARE_REF - PcB
    if np.any(Pc < -1e-12):
        raise ValueError("Negative car share: xi too large for this bike share")
    G_ref = TRIPS_PER_YEAR * CAR_SHARE_REF * CAR_DISTANCE_KM / CAR_OCCUPANCY
    G = G_ref * (CAR_SHARE_REF - r_BC * PcB) / CAR_SHARE_REF
    dC = CAR_DISTANCE_KM * (CAR_SHARE_REF - r_BC * PcB) / np.maximum(Pc, 1e-12)
    return Pc, dC, G

def emission_matrix(xi, r_BC):
    # E[t, s] = tCO2 in decision year t if the cumulative budget is at lookup state s.
    _, _, G = car_model(P, xi, r_BC)
    return np.outer(emission_factor, G) / 1e6

REFERENCE_ANNUAL = emission_matrix(1.0, 1.0)[:, 0]    # no new investment: same for every parameter set
REFERENCE_CUMULATIVE = float(REFERENCE_ANNUAL.sum())
CO2_LIMIT = (1 - CO2_REDUCTION_TARGET) * REFERENCE_CUMULATIVE
print(f"Reference car emissions: {REFERENCE_ANNUAL[0]:,.0f} tCO2 in {years[0]}, "
      f"{REFERENCE_CUMULATIVE:,.0f} tCO2 cumulated {years[0]}–{years[-1]}")
print(f"Target ({CO2_REDUCTION_TARGET:.0%} less): cumulative ≤ {CO2_LIMIT:,.0f} tCO2")

Reference car emissions: 4,615 tCO2 in 2027, 110,769 tCO2 cumulated 2027–2050
Target (10% less): cumulative ≤ 99,692 tCO2


## 4. Optimizer: exact dynamic programming over the MLCC states

State = lookup index $s_t$ reached by year $t$ (cumulative allocation $b[s_t]$); each year the index can grow by $0\ldots K_{max}$ steps (annual cap). The CO$_2$ constraint is handled with a Lagrange multiplier $\lambda$ (€/tCO$_2$): for a given $\lambda$, the DP minimises $\sum_t \text{disc}_t\,(b[s_t]-b[s_{t-1}]) + \lambda\sum_t E_t(s_t)$ exactly, and a bisection finds the smallest $\lambda$ that satisfies the cumulative target ($\lambda$ ≈ **marginal abatement cost**). Because the problem is discrete, the Lagrange solution can overshoot the target; a final trimming step removes the latest lookup steps while the target still holds. The remaining slack is reported.

In [4]:
def dp_path(lam, E, s_min_final=0):
    S = len(b)
    idx = np.arange(S)
    V = np.where(idx >= s_min_final, 0.0, np.inf)        # terminal condition (optional share target)
    choice = np.zeros((T, S), dtype=int)
    for t in range(T - 1, -1, -1):
        W = discount[t] * b + lam * E[t] + V
        best = W.copy(); arg = idx.copy()                 # k = 0: no new allocation this year
        for k in range(1, K_MAX + 1):
            cand = np.full(S, np.inf); cand[:S - k] = W[k:]
            better = cand < best
            best[better] = cand[better]; arg[better] = idx[better] + k
        V = best - discount[t] * b
        choice[t] = arg
    path, s = [], 0
    for t in range(T):
        s = int(choice[t, s]); path.append(s)
    return np.array(path)

def evaluate(path, E):
    cum = b[path]
    annual = np.diff(np.r_[0.0, cum])
    return dict(cumulative=float(E[np.arange(T), path].sum()),
                discounted_cost=float((discount * annual).sum()),
                total_allocated=float(cum[-1]))

def trim(path, E, limit, s_min=0):
    # The Lagrange solution can overshoot the target (duality gap). Remove the latest lookup
    # steps one by one while the CO2 target (and the optional terminal share) still holds.
    path = path.copy()
    rows = np.arange(T)
    improved = True
    while improved:
        improved = False
        for t in np.flatnonzero(np.diff(np.r_[0, path]) > 0)[::-1]:
            cand = path.copy(); cand[t:] -= 1
            if cand[-1] >= s_min and E[rows, cand].sum() <= limit + 1e-6:
                path, improved = cand, True
                break
    return path

def solve(xi, r_BC, reduction=CO2_REDUCTION_TARGET, terminal_share=TERMINAL_SHARE_TARGET):
    E = emission_matrix(xi, r_BC)
    limit = (1 - reduction) * REFERENCE_CUMULATIVE
    s_min = 0 if terminal_share is None else int(np.searchsorted(P, terminal_share - 1e-9))
    if s_min >= len(b):
        return dict(feasible=False, reason="terminal share above the lookup range")
    max_path = dp_path(1e12, E, s_min)                    # emissions dominate: maximum effort
    best = evaluate(max_path, E)
    max_reduction = 1 - best["cumulative"] / REFERENCE_CUMULATIVE
    if best["cumulative"] > limit + 1e-6:
        return dict(feasible=False, path=max_path, max_reduction=max_reduction, E=E,
                    reason=f"max achievable reduction {max_reduction:.2%} < target {reduction:.0%}")
    lo, hi = 0.0, 1.0
    path = dp_path(0.0, E, s_min)
    if evaluate(path, E)["cumulative"] <= limit + 1e-6:
        hi = 0.0
    else:
        while evaluate(dp_path(hi, E, s_min), E)["cumulative"] > limit + 1e-6:
            lo, hi = hi, hi * 10
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            if evaluate(dp_path(mid, E, s_min), E)["cumulative"] > limit + 1e-6:
                lo = mid
            else:
                hi = mid
        path = dp_path(hi, E, s_min)
    path = trim(path, E, limit, s_min)
    res = evaluate(path, E)
    return dict(feasible=True, path=path, E=E, shadow_price=hi, max_reduction=max_reduction,
                slack_t=limit - res["cumulative"], **res)

def plan_table(sol, xi, r_BC, name):
    path, E = sol["path"], sol["E"]
    cum = b[path]
    Pc, dC, G = car_model(P[path], xi, r_BC)
    annual_co2 = E[np.arange(T), path]
    df = pd.DataFrame({
        "scenario": name, "year": years,
        "annual_allocated_eur": np.diff(np.r_[0.0, cum]),
        "discounted_cost_eur": discount * np.diff(np.r_[0.0, cum]),
        "cumulative_allocated_eur": cum,
        "normalized_network_length": N[path], "weighted_lcc_normalized": C[path],
        "bike_share_percent": P[path], "car_share_percent": 100 * Pc,
        "car_distance_km": dC, "car_vkm": G,
        "annual_co2_t": annual_co2, "reference_co2_t": REFERENCE_ANNUAL,
        "cumulative_co2_t": np.cumsum(annual_co2),
        "reference_cumulative_co2_t": np.cumsum(REFERENCE_ANNUAL),
    })
    first = pd.DataFrame([{"scenario": name, "year": START_YEAR, "annual_allocated_eur": 0.0,
                           "discounted_cost_eur": 0.0, "cumulative_allocated_eur": 0.0,
                           "normalized_network_length": N[0], "weighted_lcc_normalized": C[0],
                           "bike_share_percent": P0, "car_share_percent": 100 * CAR_SHARE_REF,
                           "car_distance_km": CAR_DISTANCE_KM, "cumulative_co2_t": 0.0,
                           "reference_cumulative_co2_t": 0.0}])
    return pd.concat([first, df], ignore_index=True)

## 5. Feasibility and least-cost plan for every parameter set

`max_reduction` is obtained by investing as early as allowed by the cap, up to the end of the lookup (0.5 B$_{max}$). The abatement cost is the discounted cost divided by the avoided tonnes (compare with the *Coût d'abattement* table of `bicycle_model.tex`).

In [5]:
rows, plans = [], {}
for name, prm in PARAMETER_SETS.items():
    sol = solve(**prm)
    row = {"parameter_set": name, **prm, "max_reduction_percent": 100 * sol["max_reduction"],
           "feasible": sol["feasible"]}
    if sol["feasible"]:
        avoided = REFERENCE_CUMULATIVE - sol["cumulative"]
        plans[name] = plan_table(sol, prm["xi"], prm["r_BC"], name)
        row.update(total_allocated_meur=sol["total_allocated"] / 1e6,
                   discounted_cost_meur=sol["discounted_cost"] / 1e6,
                   achieved_reduction_percent=100 * avoided / REFERENCE_CUMULATIVE,
                   bike_share_end_percent=float(P[sol["path"][-1]]),
                   abatement_cost_eur_per_t=sol["discounted_cost"] / avoided,
                   marginal_cost_eur_per_t=sol["shadow_price"], slack_t=sol["slack_t"])
    else:
        row["note"] = sol["reason"]
    rows.append(row)
feasibility = pd.DataFrame(rows)
display(feasibility.round(3))
feasibility.to_csv(OUTPUT_DIR / "feasibility_and_cost_by_parameter_set.csv", index=False)

,parameter_set,xi,r_BC,max_reduction_percent,feasible,note,total_allocated_meur,discounted_cost_meur,achieved_reduction_percent,bike_share_end_percent,abatement_cost_eur_per_t,marginal_cost_eur_per_t,slack_t
0,Conservative,0.30,0.27,4.314,False,max achievable reduction 4.31% < target 10%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Intermediate,0.50,0.35,9.321,False,max achievable reduction 9.32% < target 10%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Optimistic,0.65,0.40,13.848,True,NaN,9.427,8.609,10.012,32.960,776.323,2044.095,12.907
3,Original notebook,1.00,1.00,53.261,True,NaN,0.193,0.187,10.864,25.536,15.554,15.554,957.380


## 6. Detailed plan for the main scenario

In [6]:
prm = PARAMETER_SETS[SCENARIO]
main = solve(**prm)
if main["feasible"]:
    plan = plans[SCENARIO]
    title = f"{SCENARIO}: least-cost plan for −{CO2_REDUCTION_TARGET:.0%} cumulative CO2"
else:
    # Target unreachable: show the maximum-effort plan and what it achieves.
    print(f"'{SCENARIO}': target infeasible — {main['reason']}. Showing the maximum-effort plan.")
    plan = plan_table(main, prm["xi"], prm["r_BC"], f"{SCENARIO} (maximum effort)")
    title = f"{SCENARIO}: maximum-effort plan (−{main['max_reduction']:.1%} achievable)"
plan.to_csv(OUTPUT_DIR / f"plan_{SCENARIO.replace(' ', '_').lower()}.csv", index=False)
print(title)
show = plan[plan.year.isin([2026, 2027, 2030, 2035, 2040, 2045, 2050])]
display(show[["year", "annual_allocated_eur", "cumulative_allocated_eur", "bike_share_percent",
              "car_share_percent", "car_distance_km", "annual_co2_t", "cumulative_co2_t",
              "reference_cumulative_co2_t"]].round(2))

'Intermediate': target infeasible — max achievable reduction 9.32% < target 10%. Showing the maximum-effort plan.
Intermediate: maximum-effort plan (−9.3% achievable)


,year,annual_allocated_eur,cumulative_allocated_eur,bike_share_percent,car_share_percent,car_distance_km,annual_co2_t,cumulative_co2_t,reference_cumulative_co2_t
0,2026,0.00,0.00,22.82,25.00,8.00,NaN,0.00,0.00
1,2027,1970352.81,1970352.81,27.11,22.85,8.49,4476.66,4476.66,4615.38
4,2030,1970352.81,7881411.23,32.15,20.33,9.19,4313.84,17549.28,18461.54
9,2035,1970352.81,17733175.28,35.31,18.76,9.73,4211.96,38772.29,41538.46
14,2040,1970352.81,27584939.32,37.56,17.63,10.17,4139.10,59610.32,64615.38
19,2045,1777180.96,37243531.52,39.53,16.64,10.61,4075.45,80109.23,87692.31
24,2050,0.00,38595734.43,39.79,16.51,10.67,4067.10,100444.72,110769.23


## 7. Pathways of the feasible least-cost plans

In [7]:
colors = {"Conservative": "#b91c1c", "Intermediate": "#ea580c", "Optimistic": "#16a34a",
          "Original notebook": "#2563eb"}
to_plot = dict(plans)
if not main["feasible"]:
    to_plot[f"{SCENARIO} (maximum effort)"] = plan
for column, scale, label in [("annual_allocated_eur", 1e6, "Annual allocation (€ million)"),
                             ("cumulative_allocated_eur", 1e6, "Cumulative allocation (€ million)"),
                             ("bike_share_percent", 1, "Bike (two-wheel) mode share (%)"),
                             ("cumulative_co2_t", 1, "Cumulative car CO2 since 2027 (t)")]:
    fig = go.Figure()
    for name, df in to_plot.items():
        base = name.replace(" (maximum effort)", "")
        fig.add_trace(go.Scatter(x=df.year, y=df[column] / scale, mode="lines+markers", name=name,
                                 line=dict(color=colors.get(base, "#555"), width=2.5,
                                           dash="dot" if "maximum" in name else "solid")))
    if column == "cumulative_co2_t":
        ref = next(iter(to_plot.values()))
        fig.add_trace(go.Scatter(x=ref.year, y=ref.reference_cumulative_co2_t, name="Reference (no investment)",
                                 line=dict(color="black", dash="dash")))
        fig.add_hline(y=CO2_LIMIT, line_dash="dot", annotation_text=f"{END_YEAR} target: −{CO2_REDUCTION_TARGET:.0%}")
    if column == "annual_allocated_eur":
        fig.add_hline(y=ANNUAL_ALLOCATION_CAP_EUR / 1e6, line_dash="dot", annotation_text="annual cap")
    fig.update_layout(title=f"Grenoble {START_YEAR}–{END_YEAR} — {label}", xaxis_title="Year",
                      yaxis_title=label, template="plotly_white", height=450)
    fig.show()

## 8. Backcasting curve: cost of each cumulative-CO$_2$ reduction target

Least discounted cost to reach a given cumulative reduction by END_YEAR, for each parameter set (only targets that are feasible within the lookup and the annual cap are shown). Useful to choose a realistic target or to see where cycling infrastructure alone saturates.

In [8]:
curve_rows = []
for name, prm in PARAMETER_SETS.items():
    top = solve(**prm, reduction=0.0)["max_reduction"]
    for target in np.linspace(0.005, top, 25):
        sol = solve(**prm, reduction=float(target))
        if sol["feasible"]:
            curve_rows.append({"parameter_set": name, "target_reduction_percent": 100 * target,
                               "discounted_cost_meur": sol["discounted_cost"] / 1e6,
                               "total_allocated_meur": sol["total_allocated"] / 1e6})
curve = pd.DataFrame(curve_rows)
curve.to_csv(OUTPUT_DIR / "cost_vs_reduction_target.csv", index=False)
fig = go.Figure()
for name, df in curve.groupby("parameter_set", sort=False):
    fig.add_trace(go.Scatter(x=df.target_reduction_percent, y=df.discounted_cost_meur, mode="lines+markers",
                             name=name, line=dict(color=colors.get(name, "#555"), width=2.5)))
fig.add_vline(x=100 * CO2_REDUCTION_TARGET, line_dash="dot", annotation_text=f"{CO2_REDUCTION_TARGET:.0%} target")
fig.update_layout(title=f"Grenoble — least discounted cost vs cumulative CO2 reduction ({years[0]}–{END_YEAR})",
                  xaxis_title="Cumulative CO2 reduction vs reference (%)",
                  yaxis_title="Discounted cost (€ million)", template="plotly_white", height=500)
fig.show()

## 9. Saved files

In [9]:
pd.concat(plans.values(), ignore_index=True).to_csv(OUTPUT_DIR / "plans_all_feasible_parameter_sets.csv", index=False)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(p.resolve())

\results\backcasting\co2_target\cost_vs_reduction_target.csv
\results\backcasting\co2_target\feasibility_and_cost_by_parameter_set.csv
\results\backcasting\co2_target\plan_intermediate.csv
\results\backcasting\co2_target\plans_all_feasible_parameter_sets.csv


### Interpretation and limits
* The bike share is the **INSEE "Deux-roues total"** calibration (two-wheelers), and responds instantly to the cumulative allocation (no construction or adoption delay).
* Only the infrastructure effect is modelled; other policies ($u_p$, $p\neq b$ in `bicycle_model.tex`) and the reference trends $\bar P_C(t)$, $\bar d_C(t)$ are kept constant.
* The emission factor is constant unless `EMISSION_FACTOR_END` is changed; a cleaner fleet lowers the CO$_2$ value of later bike trips and therefore pushes investments earlier.
* The plans are restricted to the MLCC lookup range (0–0.5 B$_{max}$) and to MLCC's order of upgrades; a different upgrade policy could reach other shares for the same budget.